# Name That LLM — 18 分类（识别文本由哪个大模型生成）

**整体方案（全部在本 notebook 内完成，按顺序运行即可）**

| 步骤 | 内容 | 设备 |
|---|---|---|
| 0 | 配置、读数据、统一的分组 5 折 | CPU |
| 1 | TF-IDF（字符 1-5 元 + 词 1-2 元）+ 逻辑回归 | CPU |
| 2 | 微调 DeBERTa-v3-large 分类器（5 折，可断点续跑） | **GPU** |
| 3 | 融合各模型的 OOF 概率（按 logloss 搜索权重） | CPU |
| 4 | 可选：同一主题组内"一对一分配"后处理 | CPU |
| 5 | 生成 `submission.parquet` / `submission.csv` | CPU |

**关键设计**
- 训练集里 150 个 prompt 每个都被 18 个模型各答一次 → 交叉验证必须 **按 `prompt_id` 分组**，否则同题文本泄漏、分数虚高。
- Kaggle 使用方法：右侧 *Accelerator* 选 GPU（T4×2 / P100）；如果比赛要求关闭网络，需通过 *Add Input → Models* 添加 `deberta-v3-large`，并在配置单元里把 `MODEL` 改成它的路径。
- 第 2 步每折结束都会存盘，会话中断后重新运行会自动跳过已完成的折。
- 没有 GPU 时第 2 步会自动跳过，只用 TF-IDF 也能生成提交文件。

## 0. 配置与数据

In [ ]:
import os, re, glob, math, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# ------------------------- 路径（自动识别 Kaggle / 本地） -------------------------
def find_data_dir():
    """依次尝试：环境变量 -> 比赛默认目录 -> 本地上传目录 -> /kaggle/input 下任意含 train.csv 的目录"""
    for p in [os.environ.get("DATA_DIR"), "/kaggle/input/competitions/name-that-llm", "/kaggle/input/name-that-llm"]:
        if p and (Path(p) / "train.csv").exists():
            return Path(p)
    hits = glob.glob("/kaggle/input/*/train.csv")
    if hits:
        return Path(hits[0]).parent
    raise FileNotFoundError("找不到 train.csv，请手动设置 DATA")

DATA = find_data_dir()
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("OUT_DIR", "./output"))
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA, "| OUT:", OUT)

# ------------------------- 可调参数 -------------------------
N_FOLDS = 5                                  # 分组交叉验证折数（所有模型必须一致）
RUN_TRANSFORMER = None                       # None=有 GPU 才跑；True/False 可强制开关
MODEL = "/kaggle/input/models/romanchirkov1/microsoftdeberta-v3-large/transformers/default/1"         # 离线时改成 /kaggle/input/.../deberta-v3-large 的本地路径
TAG = Path(MODEL.rstrip("/")).name           # 结果文件前缀
MAX_LEN = 400                                # 最长训练文本约 274 词，400 个 token 基本不截断
EPOCHS, BS, ACC = 4, 4, 4                    # epoch 数；batch size；梯度累积（等效 batch = BS*ACC）
#BS, ACC = 4, 4          # 每次只放 4 条，累积 4 次，等效 batch 仍是 16
GRAD_CKPT = True   
LR = 1.5e-5                                  # large 模型 1~2e-5；base 模型可用 3e-5
GRAD_CKPT = True                            # 显存不足时改 True（更慢但省显存）
ASSIGN = "auto"     

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder

tr = pd.read_csv(DATA / "train.csv")         # 列：id, prompt_id, text, model
te = pd.read_csv(DATA / "test.csv")          # 列：id, text
le = LabelEncoder().fit(tr.model)            # 18 个模型名 -> 0..17（所有步骤共用同一编码）
y = le.transform(tr.model)
N, K = len(tr), len(le.classes_)

# 按 prompt_id 分组的 5 折：同一个 prompt 的 18 条回答只会出现在同一折
folds = list(GroupKFold(N_FOLDS).split(tr, y, tr.prompt_id))
print(f"train {tr.shape}, test {te.shape}, 类别数 {K}")
print(tr.model.value_counts().describe()[["min", "max"]].to_dict(), "（各类别样本数）")
print("词数分布:", tr.text.str.split().str.len().describe()[["min", "50%", "max"]].to_dict())

## 1. TF-IDF + 逻辑回归（强基线）
不同模型的口头禅、标点（破折号、冒号）、排版和词汇习惯差异很大，字符 n-gram 对此很敏感。
本步骤同时产出：**OOF 概率**（用于线下评估和融合权重）和 **测试集概率**（用全部训练数据重新拟合）。

In [ ]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def tfidf_fit_predict(train_text, train_y, *pred_texts):
    """在 train_text 上拟合 TF-IDF + LR，返回对每个 pred_texts 的类别概率"""
    char = TfidfVectorizer(analyzer="char", ngram_range=(1, 5), sublinear_tf=True, min_df=3,
                           lowercase=False, max_features=300000)   # 保留大小写与标点
    word = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True, min_df=2,
                           lowercase=False, token_pattern=r"(?u)\b\w+\b|[^\w\s]")  # 标点也当作 token
    Xa = hstack([char.fit_transform(train_text), word.fit_transform(train_text)]).tocsr()
    clf = LogisticRegression(C=20, max_iter=300).fit(Xa, train_y)
    return [clf.predict_proba(hstack([char.transform(t), word.transform(t)]).tocsr()) for t in pred_texts]

oof_tfidf = np.zeros((N, K))
for k, (tri, vai) in enumerate(folds):
    oof_tfidf[vai] = tfidf_fit_predict(tr.text.iloc[tri], y[tri], tr.text.iloc[vai])[0]
    print(f"fold {k}: acc {(oof_tfidf[vai].argmax(1) == y[vai]).mean():.4f}")
print("TF-IDF OOF acc:", round((oof_tfidf.argmax(1) == y).mean(), 4), "（随机猜测约 0.056）")

test_tfidf = tfidf_fit_predict(tr.text, y, te.text)[0]       # 全量训练 -> 测试集概率
oofs, tests = {"tfidf": oof_tfidf}, {"tfidf": test_tfidf}    # 收集各模型结果，供步骤 3 融合

## 2. 微调 DeBERTa-v3-large（GPU）
- fp16 混合精度（T4/P100 不支持 bf16），动态 padding，余弦学习率 + 10% 预热，梯度裁剪 1.0，标签平滑 0.05。
- 每个 epoch 在验证折上评估，保留 **验证 logloss 最低** 那个 epoch 的验证/测试预测。
- 每折结果存为 `{TAG}_fold{k}.npz`；重新运行会自动跳过已完成的折（适合 Kaggle 会话中断）。
- 想再加别的模型（如 Qwen 的 LoRA 分类器），只要产出同格式的 npz 并换一个 `TAG`，步骤 3 会自动融合。

In [ ]:
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except Exception as e:                       # 没装 torch / 环境异常时直接跳过本步骤
    print("torch 不可用，跳过 Transformer:", repr(e)[:100]); HAS_GPU = False
DO_TRANSFORMER = HAS_GPU if RUN_TRANSFORMER is None else bool(RUN_TRANSFORMER)
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "无", "| 运行 Transformer:", DO_TRANSFORMER)

if DO_TRANSFORMER:
    import torch.nn.functional as F
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_cosine_schedule_with_warmup
    dev = "cuda"
    tok = AutoTokenizer.from_pretrained(MODEL)
    encode = lambda s: tok(list(s), truncation=True, max_length=MAX_LEN)["input_ids"]   # 只分词不补齐
    ids_tr, ids_te = encode(tr.text), encode(te.text)

    def make_batch(ids, chunk):
        """把一批长短不一的序列补齐到该批最大长度（动态 padding，省显存省时间）"""
        L = max(len(ids[j]) for j in chunk)
        x = np.full((len(chunk), L), tok.pad_token_id, dtype=np.int64)
        m = np.zeros((len(chunk), L), dtype=np.int64)
        for r, j in enumerate(chunk):
            x[r, :len(ids[j])] = ids[j]; m[r, :len(ids[j])] = 1
        return torch.from_numpy(x).to(dev), torch.from_numpy(m).to(dev)

    @torch.no_grad()
    def predict(model, ids, bs=32):
        """按长度排序后批量推理（更快），再还原为原顺序，返回 softmax 概率"""
        model.eval(); order = np.argsort([len(s) for s in ids]); out = np.zeros((len(ids), K), np.float32)
        for i in range(0, len(order), bs):
            ch = order[i:i + bs]; x, m = make_batch(ids, ch)
            with torch.autocast(dev, dtype=torch.float16):
                logits = model(input_ids=x, attention_mask=m).logits
            out[ch] = F.softmax(logits.float(), -1).cpu().numpy()
        return out

    yt = torch.from_numpy(y).to(dev)

    def run_fold(k, tri, vai):
        """训练第 k 折，保存验证集/测试集概率"""
        torch.manual_seed(k); rng = np.random.default_rng(k)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=K).to(dev)
        if GRAD_CKPT: model.gradient_checkpointing_enable()
        no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]       # 偏置和 LayerNorm 不做权重衰减
        groups = [{"params": [p for n, p in model.named_parameters() if not any(t in n for t in no_decay)], "weight_decay": 0.01},
                  {"params": [p for n, p in model.named_parameters() if any(t in n for t in no_decay)], "weight_decay": 0.0}]
        opt = torch.optim.AdamW(groups, lr=LR)
        n_batch = math.ceil(len(tri) / BS); total_steps = EPOCHS * math.ceil(n_batch / ACC)
        sched = get_cosine_schedule_with_warmup(opt, int(0.1 * total_steps), total_steps)
        scaler = torch.cuda.amp.GradScaler()                               # fp16 需要 loss scaling
        best_ll, best_val, best_test, t0 = 1e9, None, None, time.time()
        for ep in range(EPOCHS):
            model.train(); order = rng.permutation(tri)                    # 每个 epoch 重新打乱
            for s in range(n_batch):
                ch = order[s * BS:(s + 1) * BS]; x, m = make_batch(ids_tr, ch)
                with torch.autocast(dev, dtype=torch.float16):
                    logits = model(input_ids=x, attention_mask=m).logits
                loss = F.cross_entropy(logits.float(), yt[ch], label_smoothing=0.05) / ACC
                scaler.scale(loss).backward()
                if (s + 1) % ACC == 0 or s == n_batch - 1:                 # 每 ACC 个小批更新一次参数
                    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            pv = predict(model, [ids_tr[j] for j in vai])
            ll = float(-np.log(pv[np.arange(len(vai)), y[vai]] + 1e-9).mean()); acc = float((pv.argmax(1) == y[vai]).mean())
            print(f"[fold {k}] epoch {ep+1}/{EPOCHS}  val_logloss {ll:.4f}  val_acc {acc:.4f}  ({time.time()-t0:.0f}s)", flush=True)
            if ll < best_ll:                                               # 只保留验证 logloss 最好的 epoch
                best_ll, best_val, best_test = ll, pv, predict(model, ids_te)
        np.savez(OUT / f"{TAG}_fold{k}.npz", idx=vai, val=best_val, test=best_test)
        del model, opt; torch.cuda.empty_cache()

    for k, (tri, vai) in enumerate(folds):
        if (OUT / f"{TAG}_fold{k}.npz").exists():
            print(f"fold {k} 已存在，跳过"); continue
        run_fold(k, tri, vai)

## 3. 融合
读取所有已完成 5 折的 Transformer 结果（`{TAG}_fold*.npz`），与 TF-IDF 一起，用 **加权几何平均** 融合，
权重通过最小化 OOF logloss 搜索（模型很少、参数极少，过拟合风险小）。

In [ ]:
# 读取磁盘上所有"5 折齐全"的 Transformer 结果
for tag in sorted({re.sub(r"_fold\d\.npz$", "", Path(p).name) for p in glob.glob(str(OUT / "*_fold*.npz"))}):
    files = sorted(glob.glob(str(OUT / f"{tag}_fold*.npz")))
    if len(files) < N_FOLDS:
        print(f"跳过 {tag}：只完成 {len(files)}/{N_FOLDS} 折"); continue
    o, t = np.zeros((N, K)), np.zeros((len(te), K))
    for p in files:
        z = np.load(p); o[z["idx"]] = z["val"]; t += z["test"] / len(files)   # 测试概率 = 各折平均
    oofs[tag], tests[tag] = o, t

logloss = lambda p: float(-np.log(p[np.arange(N), y] + 1e-9).mean())
for name, o in oofs.items():
    print(f"{name:28s} acc {(o.argmax(1) == y).mean():.4f}   logloss {logloss(o):.4f}")

def combine(weights, probs):
    """加权几何平均：softmax(sum_i w_i * log p_i)"""
    z = sum(w * np.log(p + 1e-9) for w, p in zip(weights, probs)); z -= z.max(1, keepdims=True)
    e = np.exp(z); return e / e.sum(1, keepdims=True)

from scipy.optimize import minimize
names = list(oofs)
res = minimize(lambda a: logloss(combine(np.exp(a), [oofs[n] for n in names])), np.zeros(len(names)), method="Nelder-Mead")
weights = np.exp(res.x)                                          # exp 保证权重为正
print("融合权重:", {n: round(float(w), 3) for n, w in zip(names, weights)})
oof_blend  = combine(weights, [oofs[n]  for n in names])
test_blend = combine(weights, [tests[n] for n in names])
print(f"融合后 OOF  acc {(oof_blend.argmax(1) == y).mean():.4f}   logloss {logloss(oof_blend):.4f}")

## 4. 可选后处理：同主题组内的一对一分配
训练集里每个 prompt 恰有 18 条回答、对应 18 个不同模型。如果测试集也是这样，就可以：
先按**主题**把文本聚成组（每组 ≈18 条），再在组内用匈牙利算法让"每个模型恰好用一次"，整体概率最大。
- 在训练集 OOF 上验证过（TF-IDF 基线 35% → 约 45%）。
- **自动保护**：只有当测试文本确实聚成约 18 条一组、且该方法在训练集 OOF 上有提升时才启用；否则沿用 argmax。

In [ ]:
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

def assign_within_topics(texts, probs, group=18):
    """返回 (预测类别, 落在"约 group 条一组"的簇里的样本占比)"""
    pred, n = probs.argmax(1).copy(), len(texts)
    if n < 2 * group:                                    # 样本太少，不可能有分组结构
        return pred, 0.0
    # 1) 主题表示：去停用词的 TF-IDF -> SVD 降维 -> 归一化（余弦距离）
    X = TfidfVectorizer(stop_words="english", sublinear_tf=True, min_df=2, max_df=0.5).fit_transform(texts)
    Z = normalize(TruncatedSVD(min(150, n - 1), random_state=0).fit_transform(X))
    # 2) 聚成 n/18 个主题簇
    cl = AgglomerativeClustering(n_clusters=round(n / group), metric="cosine", linkage="average").fit_predict(Z)
    # 3) 簇内匈牙利分配：每个模型最多用一次，总对数概率最大
    well_formed = 0
    for c in np.unique(cl):
        idx = np.where(cl == c)[0]
        if len(idx) <= group:
            r, col = linear_sum_assignment(-np.log(probs[idx] + 1e-9)); pred[idx[r]] = col
            well_formed += len(idx) if len(idx) >= 0.8 * group else 0
    return pred, well_formed / n

# 先在训练集 OOF 上验证这个后处理到底有没有用
base_acc = (oof_blend.argmax(1) == y).mean()
tr_pred, _ = assign_within_topics(tr.text.values, oof_blend)
assign_acc = (tr_pred == y).mean()
print(f"训练集 OOF：argmax {base_acc:.4f}  ->  组内分配 {assign_acc:.4f}")

test_pred = test_blend.argmax(1)                         # 默认：直接取概率最大的类别
if ASSIGN != "0":
    te_pred, frac = assign_within_topics(te.text.values, test_blend)
    print(f"测试集：{frac:.0%} 的样本落在约 18 条一组的簇里")
    if frac >= 0.6 and assign_acc > base_acc:
        test_pred = te_pred; print("-> 已启用组内分配")
    else:
        print("-> 未启用（测试集看起来没有分组结构，或训练集上无提升）")

## 5. 生成提交文件

In [ ]:
labels = le.classes_[test_pred]                          # 编码 -> 模型名
sample_path = DATA / "sample_submission.parquet"
# 严格沿用样例提交的 id 顺序和列名（id, model）
sub = pd.read_parquet(sample_path)[["id"]] if sample_path.exists() else te[["id"]].copy()
sub["model"] = sub["id"].map(dict(zip(te.id, labels)))
assert sub["model"].notna().all() and len(sub) == len(te)
sub.to_parquet(OUT / "submission.parquet", index=False)  # 与样例同格式
sub.to_csv(OUT / "submission.csv", index=False)          # 备份：csv 版本
print("已保存:", OUT / "submission.parquet", sub.shape)
sub.head()